# Expérimentations SOM (Self-Organizing Map)

Ce notebook présente les expérimentations de la carte de Kohonen (SOM) :
1. **Visualisation jouet 2D** : Comment la grille se déplie et épouse la distribution de données bidimensionnelles.
2. **Expérimentation MNIST Sandbox** : Entraînement sur les images MNIST, tracé des prototypes de neurones et cartographie des classes majoritaires.

## Part 1. Visualisation jouet 2D : La grille W épousant les données

Dans un espace de données en 2D, nous pouvons tracer à la fois les points de données originaux et les poids $W$ des neurones. En reliant les neurones voisins sur la grille (horizontalement et verticalement), on observe visuellement comment la grille Kohonen se déforme topologiquement pour recouvrir l'espace des données.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))

import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.som import SOM
from src.dataset import load_mnist_dataset
from src.helper import extract_full_dataset
from src.metrics import compression_report

In [2]:
# 1. Génération de données uniformes dans un carré 2D [-1, 1]x[-1, 1]
rng = np.random.default_rng(42)
X_toy = rng.uniform(-1.0, 1.0, (1000, 2))

# 2. Entraînement d'une SOM sur une grille 5x5
grid_shape_toy = (5, 5)
som_toy = SOM(grid_shape=grid_shape_toy, alpha=0.6, gamma=1.5, n_iterations=3000, random_state=42)
som_toy.fit(X_toy)

In [3]:
# 3. Visualisation de la grille qui épouse la forme des données
H, W = grid_shape_toy
# Reshape de self.W en (H, W, 2) pour indexer facilement par (row, col) sur la grille
W_grid = som_toy.W.reshape(H, W, 2)

fig = go.Figure()

# Ajout des points de données originaux (en gris clair)
fig.add_trace(go.Scatter(
    x=X_toy[:, 0], y=X_toy[:, 1],
    mode='markers',
    marker=dict(size=4, color='rgba(150, 150, 150, 0.4)'),
    name='Données'
))

# Tracé des lignes horizontales reliant les neurones voisins
for r in range(H):
    fig.add_trace(go.Scatter(
        x=W_grid[r, :, 0], y=W_grid[r, :, 1],
        mode='lines+markers',
        line=dict(color='red', width=2),
        marker=dict(size=6, color='red'),
        showlegend=False
    ))

# Tracé des lignes verticales reliant les neurones voisins
for c in range(W):
    fig.add_trace(go.Scatter(
        x=W_grid[:, c, 0], y=W_grid[:, c, 1],
        mode='lines',
        line=dict(color='red', width=2),
        showlegend=False
    ))

fig.update_layout(
    title="Déploiement topologique de la grille SOM sur les données 2D",
    xaxis=dict(range=[-1.2, 1.2]),
    yaxis=dict(range=[-1.2, 1.2]),
    width=700, height=700,
    template="plotly_dark"
)

fig.show()

## Part 2. Expérimentation SOM sur MNIST Sandbox

In [4]:
dataloader = load_mnist_dataset(train=True, batch_size=4096, download=True, shuffle=True)
images, digit_labels = extract_full_dataset(dataloader)

N = 10000
X = images[:N].flatten(start_dim=1).numpy()
digit_labels = digit_labels[:N].numpy()

print("X shape:", X.shape)
print("Digit labels shape:", digit_labels.shape)

X shape: (10000, 784)
Digit labels shape: (10000,)


### 1. Entraînement de la SOM (Grille 10x10)

Nous choisissons une grille $10 \times 10$ ($100$ neurones prototypes) sur $10\ 000$ itérations.

In [5]:
grid_shape = (10, 10)
som = SOM(grid_shape=grid_shape, alpha=0.5, gamma=2.0, n_iterations=10000, random_state=42)
print("Entraînement de la SOM sur MNIST en cours...")
som.fit(X)
print("Entraînement terminé !")

Entraînement de la SOM sur MNIST en cours...
Entraînement terminé !


### 2. Rapport de compression

In [6]:
latent = som.encode(X)
X_reconstructed = som.decode(latent)
codebook = som.get_codebook()

report = compression_report(codebook, latent, X, X_reconstructed)
for k, v in report.items():
    print(f"{k}: {v}")

latent_nature: discrete
codebook_bytes: 313600
latent_bytes: 10000
total_compressed_bytes: 323600
original_bytes: 31360000
compression_ratio: 96.9097651421508
reconstruction_mse: 0.038544587790966034


### 3. Visualisation : Les prototypes comme grille topologique

Chaque neurone est reshape en $28 \times 28$ pixels et affiché à sa coordonnée correspondante.

In [7]:
fig = make_subplots(
    rows=grid_shape[0], cols=grid_shape[1],
    horizontal_spacing=0.01, vertical_spacing=0.01
)

for r in range(grid_shape[0]):
    for c in range(grid_shape[1]):
        neuron_idx = r * grid_shape[1] + c
        neuron_img = som.W[neuron_idx].reshape(28, 28)
        
        fig.add_trace( 
            go.Heatmap(z=neuron_img, colorscale="gray", showscale=False),
            row=r+1, col=c+1
        )

fig.update_layout(
    height=800, width=800,
    title_text="SOM : Prototypes Topologiques (Grille 10x10)",
    template="plotly_dark"
)

for r in range(1, grid_shape[0] + 1):
    for c in range(1, grid_shape[1] + 1):
        fig.update_yaxes(autorange="reversed", showticklabels=False, row=r, col=c)
        fig.update_xaxes(showticklabels=False, row=r, col=c)

fig.show()

### 4. Visualisation : Cartographie des classes majoritaires

In [8]:
neuron_classes = np.zeros(som.n_neurons)
neuron_counts = np.zeros(som.n_neurons)

for c in range(som.n_neurons):
    samples_in_c = np.where(latent.array == c)[0]
    if len(samples_in_c) > 0:
        labels_in_c = digit_labels[samples_in_c]
        counts = np.bincount(labels_in_c)
        neuron_classes[c] = np.argmax(counts)
        neuron_counts[c] = len(samples_in_c)
    else:
        neuron_classes[c] = -1

grid_classes = neuron_classes.reshape(grid_shape)
grid_counts = neuron_counts.reshape(grid_shape)
grid_display = np.where(grid_classes >= 0, grid_classes, np.nan)

fig = px.imshow(
    grid_display,
    labels=dict(x="Grille X", y="Grille Y", color="Chiffre Majoritaire"),
    x=[str(i) for i in range(grid_shape[1])],
    y=[str(i) for i in range(grid_shape[0])],
    color_continuous_scale="Viridis",
    aspect="equal",
    title="SOM : Classes Majoritaires et Activations"
)

for r in range(grid_shape[0]):
    for c in range(grid_shape[1]):
        count = int(grid_counts[r, c])
        digit = int(grid_classes[r, c]) if grid_classes[r, c] >= 0 else ""
        label_text = f"Chiffre {digit}<br>({count})" if digit != "" else ""
        fig.add_annotation(
            x=c, y=r,
            text=label_text,
            showarrow=False,
            font=dict(color="white", size=9)
        )

fig.update_layout(width=750, height=750, template="plotly_dark")
fig.show()